# Notebook 03 -- Feature Engineering

**Project:** PropertyIQ -- Production-Grade Property Valuation System

**Purpose:** Transform raw property attributes from `master_clean.csv` into 14
ML-ready features for RandomForest training. All encodings are computed on
**training data only** and then applied to validation and drift splits to
prevent data leakage.

**Inputs:** `train_baseline.csv`, `val.csv`, `drift_window.csv`,
`rent_train.csv`, `rent_drift.csv`, `hpi_macro.csv`

**Outputs:** `features_train.csv`, `features_val.csv`, `features_drift.csv`,
`features_rent_train.csv`, `features_rent_drift.csv`,
`feature_report.json`, `encodings.json`, `feature_correlations.png`

**The 14 Features:**
| # | Feature | Source | Rationale |
|---|---------|--------|-----------|
| 1 | city_tier_encoded | Ordinal | Tier-1/2/3 price premium |
| 2 | bhk | Raw | Bedroom count |
| 3 | total_sqft | Raw | Area in sqft |
| 4 | bath | Raw | Bathroom count |
| 5 | bath_per_bhk | Derived | Luxury proxy |
| 6 | sqft_per_bhk | Derived | Spaciousness metric |
| 7 | is_large_property | Derived | Luxury segment flag (>2000 sqft) |
| 8 | city_median_price_sqft | Target enc | City-level price signal |
| 9 | locality_median_price_sqft | Target enc | Strongest single driver |
| 10 | price_sqft_city_zscore | Statistical | Cross-city normalisation |
| 11 | rbi_hpi_avg | Macro | RBI house price index signal |
| 12 | hpi_tier_interaction | Interaction | Macro x Tier amplification |
| 13 | sqft_city_interaction | Interaction | Area x Tier premium |
| 14 | bhk_sqft_ratio | Derived | Density metric |

> **CRITICAL:** Features 8, 9, 10 are computed on train ONLY and applied
> to val/drift -- this prevents data leakage.

## Cell 2 -- Imports & Constants

In [ ]:
import json
import logging
import sys
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# -- Logging ----------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("PropertyIQ.NB03")

# -- Paths ------------------------------------------------------------------
PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

assert PROCESSED_DIR.exists(), f"Processed dir not found: {PROCESSED_DIR}"

# -- City tier mapping (ordinal) --------------------------------------------
TIER_MAP = {
    "Mumbai":    3, "Bengaluru": 3, "Delhi":     3,
    "Pune":      2, "Hyderabad": 2, "Chennai":   2,
    "Kolkata":   1, "Ahmedabad": 1,
}

# -- Feature constants ------------------------------------------------------
LARGE_PROPERTY_THRESHOLD = 2000   # sqft
BATH_PER_BHK_CAP = 3.0
ZSCORE_CAP = 3.0

FINAL_FEATURE_COLS = [
    "city_tier_encoded",
    "bhk",
    "total_sqft",
    "bath",
    "bath_per_bhk",
    "sqft_per_bhk",
    "is_large_property",
    "city_median_price_sqft",
    "locality_median_price_sqft",
    "price_sqft_city_zscore",
    "rbi_hpi_avg",
    "hpi_tier_interaction",
    "sqft_city_interaction",
    "bhk_sqft_ratio",
]

TARGET_SALE = "price_sqft"
TARGET_RENTAL = "rent_monthly"

logger.info("Constants initialised -- %d features defined", len(FINAL_FEATURE_COLS))

## Cell 3 -- Load All Splits

**Why:** Load every processed split from Notebook 02 and verify their shapes
and critical columns before any feature engineering begins. Failing fast on
missing data prevents silent downstream errors.

In [ ]:
try:
    df_train = pd.read_csv(PROCESSED_DIR / "train_baseline.csv")
    df_val = pd.read_csv(PROCESSED_DIR / "val.csv")
    df_drift = pd.read_csv(PROCESSED_DIR / "drift_window.csv")
    df_rent_train = pd.read_csv(PROCESSED_DIR / "rent_train.csv")
    df_rent_drift = pd.read_csv(PROCESSED_DIR / "rent_drift.csv")
    df_hpi = pd.read_csv(PROCESSED_DIR / "hpi_macro.csv")
except FileNotFoundError as exc:
    logger.error("Missing processed file: %s", exc)
    raise

print(f"\n{'=' * 50}")
print(f"  INPUTS LOADED")
print(f"{'=' * 50}")
for name, df in [("train_baseline", df_train), ("val", df_val),
                  ("drift_window", df_drift), ("rent_train", df_rent_train),
                  ("rent_drift", df_rent_drift)]:
    print(f"  {name:<16}: {len(df):>6,} rows x {len(df.columns)} cols")
print(f"{'=' * 50}\n")

# Verify critical columns
for split_name, df in [("train", df_train), ("val", df_val), ("drift", df_drift)]:
    for col in ["price_sqft", "total_sqft", "bhk", "bath", "city"]:
        assert col in df.columns, f"{split_name} missing column: {col}"
        assert df[col].isna().sum() == 0, f"{split_name}.{col} has {df[col].isna().sum()} nulls"

logger.info("All splits loaded and verified")

## Cell 4 -- Feature Engineering Functions

**Why:** Encapsulating each transformation in a function ensures consistency
across train/val/drift splits. The `compute_*` functions extract statistics
from train only; the `apply_*` functions use those statistics on any split.

In [ ]:
def add_city_tier(df: pd.DataFrame, tier_map: Dict[str, int]) -> pd.DataFrame:
    """Add city_tier_encoded column using ordinal tier mapping.

    Tier 1 = metro (Mumbai, Bengaluru, Delhi), Tier 2 = large city,
    Tier 3 = smaller city. Unknown cities default to tier 1.

    Args:
        df (pd.DataFrame): DataFrame with 'city' column.
        tier_map (Dict[str, int]): Mapping of city name to tier integer.

    Returns:
        pd.DataFrame: DataFrame with 'city_tier_encoded' column added.

    Example:
        >>> df = add_city_tier(df, TIER_MAP)
        >>> df['city_tier_encoded'].unique()
        array([3, 2, 1])
    """
    df = df.copy()
    df["city_tier_encoded"] = df["city"].map(tier_map).fillna(1).astype(int)
    unmapped = df[~df["city"].isin(tier_map)]["city"].unique()
    if len(unmapped) > 0:
        logger.warning("Unmapped cities (assigned tier 1): %s", unmapped.tolist())
    return df


def add_ratio_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add derived ratio features: bath_per_bhk, sqft_per_bhk, bhk_sqft_ratio, is_large_property.

    All derived from existing columns -- no external data needed.

    Args:
        df (pd.DataFrame): DataFrame with bhk, bath, total_sqft columns.

    Returns:
        pd.DataFrame: DataFrame with 4 new ratio columns.

    Example:
        >>> df = add_ratio_features(df)
        >>> df[['bath_per_bhk', 'sqft_per_bhk']].head()
    """
    df = df.copy()
    df["bath_per_bhk"] = (df["bath"] / df["bhk"]).clip(upper=BATH_PER_BHK_CAP)
    df["sqft_per_bhk"] = df["total_sqft"] / df["bhk"]
    df["bhk_sqft_ratio"] = df["bhk"] / df["total_sqft"] * 1000
    df["is_large_property"] = (df["total_sqft"] > LARGE_PROPERTY_THRESHOLD).astype(int)
    return df


def compute_city_encodings(train: pd.DataFrame) -> Dict[str, Any]:
    """Compute city-level encodings from TRAINING DATA ONLY.

    Returns median price_sqft per city and mean/std for z-score computation.
    These are applied to val/drift without recomputing -- preventing leakage.

    Args:
        train (pd.DataFrame): Training dataframe only.

    Returns:
        Dict with keys 'city_medians' (city->float) and 'city_stats' (city->{mean,std}).

    Example:
        >>> enc = compute_city_encodings(df_train)
        >>> enc['city_medians']['Mumbai']
        12500.0
    """
    city_medians = train.groupby("city")["price_sqft"].median().to_dict()
    city_stats = train.groupby("city")["price_sqft"].agg(["mean", "std"]).to_dict("index")
    logger.info("City encodings computed from %d train rows, %d cities",
                len(train), len(city_medians))
    return {"city_medians": city_medians, "city_stats": city_stats}


def compute_locality_encodings(train: pd.DataFrame,
                                city_medians: Dict[str, float]) -> Dict[str, float]:
    """Compute locality-level median price encodings from training data only.

    Fallback: unseen localities use city median -- never global median.

    Args:
        train (pd.DataFrame): Training dataframe only.
        city_medians (Dict[str, float]): City median prices from compute_city_encodings.

    Returns:
        Dict[str, float]: Mapping of locality name to median price_sqft.

    Example:
        >>> loc_enc = compute_locality_encodings(df_train, enc['city_medians'])
    """
    locality_medians = train.groupby("locality")["price_sqft"].median().to_dict()
    logger.info("Locality encodings computed: %d unique localities", len(locality_medians))
    return locality_medians


def apply_all_encodings(df: pd.DataFrame,
                         city_medians: Dict[str, float],
                         locality_medians: Dict[str, float],
                         city_stats: Dict[str, Dict[str, float]]) -> pd.DataFrame:
    """Apply pre-computed encodings to any split (train, val, or drift).

    Adds: city_median_price_sqft, locality_median_price_sqft,
    price_sqft_city_zscore, hpi_tier_interaction, sqft_city_interaction.

    Handles unseen localities by falling back to city median.
    Caps z-score to [-3, 3].

    Args:
        df (pd.DataFrame): Any split with city, locality, price_sqft, rbi_hpi_avg, city_tier_encoded.
        city_medians (Dict[str, float]): City median prices from train.
        locality_medians (Dict[str, float]): Locality median prices from train.
        city_stats (Dict[str, Dict[str, float]]): City mean/std from train.

    Returns:
        pd.DataFrame: DataFrame with encoding columns added.

    Example:
        >>> df_val = apply_all_encodings(df_val, enc['city_medians'], loc_enc, enc['city_stats'])
    """
    df = df.copy()

    # City median price
    global_median = np.median(list(city_medians.values()))
    df["city_median_price_sqft"] = df["city"].map(city_medians).fillna(global_median)

    # Locality median price (fallback: city median)
    df["locality_median_price_sqft"] = df["locality"].map(locality_medians)
    unseen_mask = df["locality_median_price_sqft"].isna()
    unseen_count = unseen_mask.sum()
    if unseen_count > 0:
        df.loc[unseen_mask, "locality_median_price_sqft"] = df.loc[unseen_mask, "city"].map(city_medians)
        # If city also unseen, use global median
        still_null = df["locality_median_price_sqft"].isna()
        df.loc[still_null, "locality_median_price_sqft"] = global_median
        logger.info("Filled %d unseen localities with city median fallback", unseen_count)

    # Z-score: (price_sqft - city_mean) / city_std
    df["price_sqft_city_zscore"] = df.apply(
        lambda row: (
            (row["price_sqft"] - city_stats.get(row["city"], {"mean": global_median})["mean"])
            / max(city_stats.get(row["city"], {"std": 1.0})["std"], 1.0)
        ),
        axis=1,
    ).clip(-ZSCORE_CAP, ZSCORE_CAP)

    # Interaction features
    df["hpi_tier_interaction"] = df["rbi_hpi_avg"] * df["city_tier_encoded"]
    df["sqft_city_interaction"] = df["total_sqft"] * df["city_tier_encoded"]

    return df


logger.info("Feature functions defined: 6 functions ready")

## Cell 5 -- Compute Encodings from Train Only

**Why:** Target encodings (city median, locality median, z-score stats) must be
computed **exclusively** from training data. Using val or drift data here would
leak future information into the model, invalidating drift detection results.

In [ ]:
logger.info("Computing all encodings from TRAIN data only")
logger.info("These will be applied to val and drift -- no data from val/drift used here")

encodings = compute_city_encodings(df_train)
city_medians = encodings["city_medians"]
city_stats = encodings["city_stats"]

loc_encodings = compute_locality_encodings(df_train, city_medians)

# Print city medians
print(f"\n{'=' * 50}")
print(f"  CITY MEDIAN PRICE/SQFT (from train)")
print(f"{'=' * 50}")
for city in sorted(city_medians, key=city_medians.get, reverse=True):
    print(f"  {city:<15}: Rs {city_medians[city]:>8,.0f}/sqft")
print(f"{'=' * 50}")

# Top 10 locality medians
sorted_locs = sorted(loc_encodings.items(), key=lambda x: x[1], reverse=True)
print(f"\n  Top 10 highest-value localities (from train):")
for loc, med in sorted_locs[:10]:
    print(f"    {loc:<35}: Rs {med:>8,.0f}/sqft")

# Save encodings for dashboard use
encodings_export = {
    "city_tier_map": TIER_MAP,
    "city_medians": {k: round(v, 2) for k, v in city_medians.items()},
    "city_stats": {k: {"mean": round(s["mean"], 2), "std": round(s["std"], 2)} for k, s in city_stats.items()},
    "locality_medians_count": len(loc_encodings),
    "generated_at": datetime.now().isoformat(),
}
encodings_path = OUTPUT_DIR / "encodings.json"
with open(encodings_path, "w", encoding="utf-8") as f:
    json.dump(encodings_export, f, indent=2, default=str)
logger.info("Encodings saved to %s", encodings_path)

## Cell 6 -- Apply Features to Train Split

**Why:** The train split gets all 14 features applied first so we can verify
the feature pipeline produces correct shapes and zero nulls before applying
to val and drift.

In [ ]:
df_train_feat = df_train.copy()
df_train_feat = add_city_tier(df_train_feat, TIER_MAP)
df_train_feat = add_ratio_features(df_train_feat)
df_train_feat = apply_all_encodings(df_train_feat, city_medians, loc_encodings, city_stats)

# Verify all 14 features present
missing = set(FINAL_FEATURE_COLS) - set(df_train_feat.columns)
assert len(missing) == 0, f"Missing features: {missing}"

# Verify no nulls
null_counts = df_train_feat[FINAL_FEATURE_COLS].isna().sum()
assert null_counts.sum() == 0, f"Null features in train:\n{null_counts[null_counts > 0]}"

# Print summary
stats_df = df_train_feat[FINAL_FEATURE_COLS].describe().T[["mean", "std", "min", "max"]]
print(f"\n{'=' * 65}")
print(f"  TRAIN FEATURES -- SUMMARY")
print(f"{'=' * 65}")
print(f"  Rows     : {len(df_train_feat):,}")
print(f"  Features : {len(FINAL_FEATURE_COLS)}")
print(f"  Target   : {TARGET_SALE}")
print(f"\n  Feature stats (train):")
col_w = 28
print(f"  {'Feature':<{col_w}} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print(f"  {'-' * (col_w + 44)}")
for feat in FINAL_FEATURE_COLS:
    row = stats_df.loc[feat]
    print(f"  {feat:<{col_w}} {row['mean']:>10.2f} {row['std']:>10.2f} {row['min']:>10.2f} {row['max']:>10.2f}")
print(f"{'=' * 65}\n")

logger.info("Train features applied: %d rows x %d features", len(df_train_feat), len(FINAL_FEATURE_COLS))

## Cell 7 -- Apply Features to Val Split

**Why:** The val split uses the same encodings computed from train in Cell 5.
Unseen localities (present in val but not train) fall back to their city's
median price -- never the global median. This mirrors real-world deployment
where new localities appear over time.

In [ ]:
df_val_feat = df_val.copy()
df_val_feat = add_city_tier(df_val_feat, TIER_MAP)
df_val_feat = add_ratio_features(df_val_feat)
df_val_feat = apply_all_encodings(df_val_feat, city_medians, loc_encodings, city_stats)

# Check for unseen localities
train_locs = set(df_train["locality"].unique())
val_locs = set(df_val["locality"].unique())
new_locs_val = val_locs - train_locs
logger.info("Val has %d unseen localities (of %d total) -- using city median fallback",
            len(new_locs_val), len(val_locs))

# Verify
missing_val = set(FINAL_FEATURE_COLS) - set(df_val_feat.columns)
assert len(missing_val) == 0, f"Val missing features: {missing_val}"
null_val = df_val_feat[FINAL_FEATURE_COLS].isna().sum().sum()
assert null_val == 0, f"Val has {null_val} null feature values"

print(f"  Val features: {len(df_val_feat):,} rows x {len(FINAL_FEATURE_COLS)} features")
print(f"  Unseen localities: {len(new_locs_val)} (city median fallback applied)")

## Cell 8 -- Apply Features to Drift Split

**Why:** The drift split is processed identically to val -- same train-derived
encodings. After applying features, we compare train vs drift distributions
to confirm that drift is visible in engineered features, not just raw prices.
This table is visual proof for presentations and model governance.

In [ ]:
df_drift_feat = df_drift.copy()
df_drift_feat = add_city_tier(df_drift_feat, TIER_MAP)
df_drift_feat = add_ratio_features(df_drift_feat)
df_drift_feat = apply_all_encodings(df_drift_feat, city_medians, loc_encodings, city_stats)

# Verify
drift_locs = set(df_drift["locality"].unique())
new_locs_drift = drift_locs - train_locs
logger.info("Drift has %d unseen localities (of %d total)", len(new_locs_drift), len(drift_locs))

missing_drift = set(FINAL_FEATURE_COLS) - set(df_drift_feat.columns)
assert len(missing_drift) == 0, f"Drift missing features: {missing_drift}"
null_drift = df_drift_feat[FINAL_FEATURE_COLS].isna().sum().sum()
assert null_drift == 0, f"Drift has {null_drift} null feature values"

# Feature distribution shift table
feature_drift_shifts = {}
print(f"\n{'=' * 60}")
print(f"  FEATURE DISTRIBUTION SHIFT: TRAIN vs DRIFT")
print(f"{'=' * 60}")
print(f"  {'Feature':<28} {'Train':>8} {'Drift':>8} {'Shift':>8}")
print(f"  {'-' * 56}")

for feat in FINAL_FEATURE_COLS:
    train_mean = df_train_feat[feat].mean()
    drift_mean = df_drift_feat[feat].mean()
    shift_pct = ((drift_mean - train_mean) / abs(train_mean) * 100) if abs(train_mean) > 0.01 else 0.0
    feature_drift_shifts[feat] = round(shift_pct, 2)
    sign = "+" if shift_pct >= 0 else ""
    print(f"  {feat:<28} {train_mean:>8.1f} {drift_mean:>8.1f} {sign}{shift_pct:>6.1f}%")

print(f"{'=' * 60}\n")

logger.info("Drift features applied: %d rows", len(df_drift_feat))

## Cell 9 -- Engineer Rental Features

**Why:** The rental model predicts `rent_monthly` instead of `price_sqft`.
It uses the same base features (city tier, ratios, interactions) but computes
its own locality encodings from rent_train to avoid mixing sale and rental
price signals. The target encoding uses `rent_sqft` = rent_monthly / total_sqft.

In [ ]:
try:
    # Compute rental-specific locality encodings from rent_train
    df_rent_train_feat = df_rent_train.copy()
    df_rent_drift_feat = df_rent_drift.copy()

    # Ensure rent_sqft exists
    for df_r in [df_rent_train_feat, df_rent_drift_feat]:
        if "rent_sqft" not in df_r.columns:
            df_r["rent_sqft"] = df_r["rent_monthly"] / df_r["total_sqft"]

    # Ensure rbi_hpi_avg exists in rental data (may not have been merged)
    preprocessing_report_path = OUTPUT_DIR / "preprocessing_report.json"
    if preprocessing_report_path.exists():
        with open(preprocessing_report_path, "r") as f:
            prep_report = json.load(f)
        hpi_map = prep_report.get("hpi_period_map", {})
    else:
        hpi_map = {"pre_covid": 142.65, "transition": 166.05, "post_covid": 159.16}

    for df_r in [df_rent_train_feat, df_rent_drift_feat]:
        if "rbi_hpi_avg" not in df_r.columns:
            df_r["rbi_hpi_avg"] = df_r["data_period"].map(
                {k: float(v) for k, v in hpi_map.items()}
            ).fillna(150.0)

    # Add city tier and ratio features
    df_rent_train_feat = add_city_tier(df_rent_train_feat, TIER_MAP)
    df_rent_train_feat = add_ratio_features(df_rent_train_feat)
    df_rent_drift_feat = add_city_tier(df_rent_drift_feat, TIER_MAP)
    df_rent_drift_feat = add_ratio_features(df_rent_drift_feat)

    # Compute rental locality encodings (rent_sqft median per locality, from train only)
    rent_locality_medians = df_rent_train_feat.groupby("locality")["rent_sqft"].median().to_dict()
    rent_city_medians_val = df_rent_train_feat.groupby("city")["rent_sqft"].median().to_dict()

    # For rental we map locality_median_price_sqft to rent_sqft-based locality medians
    # and city_median_price_sqft to rent-based city medians, scaled to monthly rent level
    rent_city_medians_monthly = df_rent_train_feat.groupby("city")["rent_monthly"].median().to_dict()
    rent_city_stats = df_rent_train_feat.groupby("city")["rent_sqft"].agg(["mean", "std"]).to_dict("index")

    # Apply city/locality encodings to both rental splits
    for df_name, df_r in [("rent_train", df_rent_train_feat), ("rent_drift", df_rent_drift_feat)]:
        global_rent_median = np.median(list(rent_city_medians_val.values()))
        df_r["city_median_price_sqft"] = df_r["city"].map(rent_city_medians_val).fillna(global_rent_median)
        df_r["locality_median_price_sqft"] = df_r["locality"].map(rent_locality_medians)
        unseen = df_r["locality_median_price_sqft"].isna()
        df_r.loc[unseen, "locality_median_price_sqft"] = df_r.loc[unseen, "city"].map(rent_city_medians_val)
        df_r["locality_median_price_sqft"] = df_r["locality_median_price_sqft"].fillna(global_rent_median)

        # Z-score for rental
        df_r["price_sqft_city_zscore"] = df_r.apply(
            lambda row: (
                (row.get("rent_sqft", 0) - rent_city_stats.get(row["city"], {"mean": global_rent_median})["mean"])
                / max(rent_city_stats.get(row["city"], {"std": 1.0})["std"], 1.0)
            ),
            axis=1,
        ).clip(-ZSCORE_CAP, ZSCORE_CAP)

        # Interaction features
        df_r["hpi_tier_interaction"] = df_r["rbi_hpi_avg"] * df_r["city_tier_encoded"]
        df_r["sqft_city_interaction"] = df_r["total_sqft"] * df_r["city_tier_encoded"]

    # Update the outer variables
    df_rent_train_feat = df_rent_train_feat
    df_rent_drift_feat = df_rent_drift_feat

    # Verify
    for feat_col in FINAL_FEATURE_COLS:
        for rn, rdf in [("rent_train", df_rent_train_feat), ("rent_drift", df_rent_drift_feat)]:
            assert feat_col in rdf.columns, f"{rn} missing feature: {feat_col}"
    assert TARGET_RENTAL in df_rent_train_feat.columns, f"rent_train missing target: {TARGET_RENTAL}"

    print(f"\n  Rent train features: {len(df_rent_train_feat):,} rows x {len(FINAL_FEATURE_COLS)} features")
    print(f"  Rent drift features: {len(df_rent_drift_feat):,} rows x {len(FINAL_FEATURE_COLS)} features")
    logger.info("Rental features applied to both splits")

except Exception as exc:
    logger.error("Rental feature engineering failed: %s", exc)

## Cell 10 -- Feature Correlation Analysis

**Why:** Understanding which features correlate most strongly with the target
guides model interpretation and identifies potential redundancies. The
horizontal bar chart is a presentation-ready visual for stakeholders.

In [ ]:
# Compute correlations with target on train
corr_with_target = df_train_feat[FINAL_FEATURE_COLS + [TARGET_SALE]].corr()[TARGET_SALE].drop(TARGET_SALE)
corr_sorted = corr_with_target.abs().sort_values(ascending=False)

# Print table
print(f"\n{'=' * 55}")
print(f"  FEATURE CORRELATION WITH {TARGET_SALE} (TRAIN)")
print(f"{'=' * 55}")
for feat in corr_sorted.index:
    val = corr_with_target[feat]
    bar_len = int(abs(val) * 20)
    bar = "#" * bar_len
    sign = "+" if val >= 0 else "-"
    print(f"  {feat:<30}: {sign}{abs(val):.3f}  {bar}")
print(f"{'=' * 55}\n")

# Save correlation chart
try:
    fig, ax = plt.subplots(figsize=(10, 7))
    colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in corr_with_target[corr_sorted.index]]
    ax.barh(range(len(corr_sorted)), corr_with_target[corr_sorted.index].values, color=colors)
    ax.set_yticks(range(len(corr_sorted)))
    ax.set_yticklabels(corr_sorted.index, fontsize=10)
    ax.set_xlabel("Pearson Correlation", fontsize=12)
    ax.set_title(f"Feature Correlations with {TARGET_SALE}", fontsize=14, fontweight="bold")
    ax.invert_yaxis()
    ax.axvline(x=0, color="black", linewidth=0.5)
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    chart_path = OUTPUT_DIR / "feature_correlations.png"
    fig.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info("Correlation chart saved to %s", chart_path)
except Exception as exc:
    logger.warning("Chart generation failed: %s", exc)

## Cell 11 -- Save All Feature Files & Report

**Why:** Persisting feature files ensures Notebook 04 (Model Training) can load
them directly. The `feature_report.json` captures all metadata needed for
reproducibility and model governance documentation.

In [ ]:
# Define save columns
SAVE_COLS_SALE = FINAL_FEATURE_COLS + [TARGET_SALE]
SAVE_COLS_RENTAL = [c for c in FINAL_FEATURE_COLS if c in df_rent_train_feat.columns] + [TARGET_RENTAL]

# Save sale feature files
df_train_feat[SAVE_COLS_SALE].to_csv(PROCESSED_DIR / "features_train.csv", index=False)
df_val_feat[SAVE_COLS_SALE].to_csv(PROCESSED_DIR / "features_val.csv", index=False)
df_drift_feat[SAVE_COLS_SALE].to_csv(PROCESSED_DIR / "features_drift.csv", index=False)

# Save rental feature files
df_rent_train_feat[SAVE_COLS_RENTAL].to_csv(PROCESSED_DIR / "features_rent_train.csv", index=False)
df_rent_drift_feat[SAVE_COLS_RENTAL].to_csv(PROCESSED_DIR / "features_rent_drift.csv", index=False)

# Build feature report
correlations_dict = {k: round(float(v), 4) for k, v in corr_with_target.items()}

feature_report = {
    "generated_at": datetime.now().isoformat(),
    "project": "PropertyIQ",
    "notebook": "03_feature_engineering",
    "feature_count": len(FINAL_FEATURE_COLS),
    "feature_names": FINAL_FEATURE_COLS,
    "train_rows": int(len(df_train_feat)),
    "val_rows": int(len(df_val_feat)),
    "drift_rows": int(len(df_drift_feat)),
    "rent_train_rows": int(len(df_rent_train_feat)),
    "rent_drift_rows": int(len(df_rent_drift_feat)),
    "city_tier_map": TIER_MAP,
    "city_medians": {k: round(v, 2) for k, v in city_medians.items()},
    "top_correlations": dict(sorted(correlations_dict.items(), key=lambda x: abs(x[1]), reverse=True)),
    "unseen_localities_val": int(len(new_locs_val)),
    "unseen_localities_drift": int(len(new_locs_drift)),
    "feature_drift_shifts": feature_drift_shifts,
    "encodings_saved": "outputs/encodings.json",
}

report_path = OUTPUT_DIR / "feature_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(feature_report, f, indent=2, default=str)
assert report_path.exists()

# Print final manifest
files_manifest = {
    "features_train.csv": (len(df_train_feat), len(SAVE_COLS_SALE)),
    "features_val.csv": (len(df_val_feat), len(SAVE_COLS_SALE)),
    "features_drift.csv": (len(df_drift_feat), len(SAVE_COLS_SALE)),
    "features_rent_train.csv": (len(df_rent_train_feat), len(SAVE_COLS_RENTAL)),
    "features_rent_drift.csv": (len(df_rent_drift_feat), len(SAVE_COLS_RENTAL)),
}

print(f"\n{'=' * 55}")
print(f"  FILES SAVED -- data/processed/")
print(f"{'=' * 55}")
for fname, (rows, cols) in files_manifest.items():
    print(f"  [OK] {fname:<30} {rows:>6,} rows x {cols} cols")
print(f"\n  outputs/feature_report.json saved [OK]")
print(f"  outputs/encodings.json saved [OK]")
if (OUTPUT_DIR / "feature_correlations.png").exists():
    print(f"  outputs/feature_correlations.png saved [OK]")
print(f"{'=' * 55}\n")

logger.info("Notebook 03 complete -- all feature files saved to %s", PROCESSED_DIR)